# C-V Measurement Plotter
**Sample:** 251203 | **Sweep:** -3V to +3V (0.1V step) | **Frequencies:** 1kHz, 10kHz, 100kHz, 1MHz

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import re

plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'legend.fontsize': 9,
    'figure.dpi': 120,
})

In [ ]:
DATA_DIR = Path(r'c:\Users\whdal\python')

def parse_filename(fname):
    name = fname.stem
    m = re.match(r'^\[(.+?)\]', name)
    sample = m.group(1) if m else name
    sweep = '1st' if '1st' in name else ('2nd' if '2nd' in name else 'unknown')
    return sample, sweep

FREQ_MAP = {
    '1.000k':   '1k',
    '10.000k':  '10k',
    '100.000k': '100k',
    '1.000M':   '1M',
}
FREQ_ORDER = ['V', '1k', '10k', '100k', '1M']

def load_cv_file(fpath):
    df = pd.read_csv(fpath, sep='\t', header=0, index_col=None)
    # 헤더에서 주파수 라벨을 파싱해 표준 이름으로 변환 후 순서 고정
    df.columns = ['V'] + [FREQ_MAP.get(c, c) for c in df.columns[1:]]
    df = df[FREQ_ORDER].astype(float)
    return df

data = {}

for fpath in sorted(DATA_DIR.glob('*.txt')):
    sample, sweep = parse_filename(fpath)
    df = load_cv_file(fpath)
    data[(sample, sweep)] = df
    print(f'[{sweep}] {sample}  ->  {len(df)} points  |  cols: {list(df.columns)}')

samples = sorted(set(s for s, _ in data.keys()))
print(f'\n총 {len(samples)}개 샘플:', samples)

In [ ]:
# Plot 1: 샘플별 주파수 비교 (1st sweep)
freqs   = ['1k', '10k', '100k', '1M']
flabels = ['1 kHz', '10 kHz', '100 kHz', '1 MHz']
colors  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

ncols = 2
nrows = (len(samples) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.5*nrows), squeeze=False)

for idx, sample in enumerate(samples):
    ax = axes[idx // ncols][idx % ncols]
    df = data.get((sample, '1st'))
    if df is None:
        df = data.get((sample, '2nd'))

    for freq, label, color in zip(freqs, flabels, colors):
        ax.plot(df['V'], df[freq] * 1e12, label=label, color=color, lw=1.5)

    ax.set_title(sample)
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Capacitance (pF)')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.axvline(0, color='k', lw=0.5, ls='--')

for idx in range(len(samples), nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

fig.suptitle('C-V Curves — Frequency Comparison (1st sweep)', fontsize=14, y=1.01)
fig.tight_layout()
plt.savefig(DATA_DIR / 'CV_freq_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Plot 2: Hysteresis (1st vs 2nd sweep) @ 100kHz
TARGET_FREQ  = '100k'
TARGET_LABEL = '100 kHz'

hyst_samples = [s for s in samples if (s, '1st') in data and (s, '2nd') in data]

if not hyst_samples:
    print('Hysteresis 비교 가능한 샘플 없음 (1st & 2nd 모두 필요)')
else:
    ncols = 2
    nrows = (len(hyst_samples) + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4.5*nrows), squeeze=False)

    for idx, sample in enumerate(hyst_samples):
        ax = axes[idx // ncols][idx % ncols]
        df1 = data[(sample, '1st')]
        df2 = data[(sample, '2nd')]

        ax.plot(df1['V'], df1[TARGET_FREQ] * 1e12, label='1st sweep', color='steelblue', lw=1.8)
        ax.plot(df2['V'], df2[TARGET_FREQ] * 1e12, label='2nd sweep', color='tomato', lw=1.8, ls='--')

        ax.set_title(sample)
        ax.set_xlabel('Voltage (V)')
        ax.set_ylabel('Capacitance (pF)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.5, ls='--')
        ax.axvline(0, color='k', lw=0.5, ls='--')

    for idx in range(len(hyst_samples), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(f'C-V Hysteresis @ {TARGET_LABEL}', fontsize=14, y=1.01)
    fig.tight_layout()
    plt.savefig(DATA_DIR / 'CV_hysteresis.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
# Plot 3: 모든 샘플 한 그래프 비교 (100kHz, 1st sweep)
TARGET_FREQ  = '100k'
TARGET_LABEL = '100 kHz'

cmap = cm.get_cmap('tab10', len(samples))
fig, ax = plt.subplots(figsize=(8, 5))

for i, sample in enumerate(samples):
    df = data.get((sample, '1st'))
    if df is None:
        df = data.get((sample, '2nd'))
    ax.plot(df['V'], df[TARGET_FREQ] * 1e12, label=sample, color=cmap(i), lw=1.8)

ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Capacitance (pF)')
ax.set_title(f'C-V Comparison — All Samples @ {TARGET_LABEL}')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.axvline(0, color='k', lw=0.5, ls='--')
fig.tight_layout()
plt.savefig(DATA_DIR / 'CV_sample_comparison.png', bbox_inches='tight', dpi=150)
plt.show()